# C1 Novelty 1: meta-learner comparison (GBR vs Ridge vs Random Forest vs LightGBM)
J26-DS-361 | C1 Drift Detection Engine | Sajivan K (IT23172296)

**Question:** is the Gradient Boosting Regressor a good choice for N1, or would a simpler or different model do as well?

**Fair comparison rules (same as `run_novelty1_v12.py`):**
- Same benchmark v1.2 (1,600 conditions per dataset), same inputs (N1 full: 9 detector signals + 6 prediction stats + batch_size).
- Same group-aware CV: StratifiedGroupKFold by drift_type, 5 folds x 10 repeats, seeds 0..9, one group per drift condition.
- **Exactly the same folds for every model** in each repeat, so the comparison is paired.
- No model is tuned. Each uses fixed, standard settings chosen before running (GBR keeps its original settings).
- Both targets: fixed (primary) and paired.

**Sanity check:** GBR must reproduce the recorded N1 results (UCI fixed 0.903, paired 0.657; IEEE fixed 1.010, paired 0.354).

Runtime: about 10 to 20 minutes per dataset on Colab CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install lightgbm

## 1. Settings (edit the paths only if your Drive layout differs)

In [ ]:
import glob, json, os, time
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/J26-DS-361-Datasets'
OUT_DIR = f'{ROOT}/c1_models_v12'          # results JSON is saved here
REPEATS, FOLDS = 10, 5

def find(patterns):
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None

DATASETS = {
    'uci': {
        'csv': find([f'{ROOT}/c1_v12/benchmark_uci_adult_v12.csv',
                     f'{ROOT}/**/benchmark_uci_adult_v12*.csv']),
        'targets': {'fixed': 'actual_accuracy_drop', 'paired': 'harm_paired'},
        'recorded_gbr': {'fixed': 0.903, 'paired': 0.657},
    },
    'ieee': {
        'csv': find([f'{ROOT}/c1_ieee/benchmark_ieeecis_v12.csv',
                     f'{ROOT}/c1_ieee/benchmark_ieee*v12*.csv',
                     f'{ROOT}/**/benchmark_ieee*v12*.csv']),
        'targets': {'fixed': 'actual_auc_drop', 'paired': 'harm_auc_paired'},
        'recorded_gbr': {'fixed': 1.010, 'paired': 0.354},
    },
}
for k, v in DATASETS.items():
    print(k, '->', v['csv'])
assert all(v['csv'] for v in DATASETS.values()), 'A benchmark CSV was not found: set the path by hand above.'

uci -> /content/drive/MyDrive/J26-DS-361-Datasets/c1_v12/benchmark_uci_adult_v12.csv
ieee -> /content/drive/MyDrive/J26-DS-361-Datasets/c1_ieee/benchmark_ieee_cis_v12.csv


## 2. Inputs, models and the CV loop

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

DET_V12 = ['ks_mean', 'ks_max', 'chi2_mean', 'chi2_max', 'psi_mean', 'psi_max', 'mmd', 'lsdd', 'clf_auc']
CONF = ['conf_mean', 'conf_std', 'conf_pct_lt_06']
POS = ['mean_pos_prob', 'pos_prob_std', 'pct_pos_prob_lt_06']
INPUTS = DET_V12 + CONF + POS + ['batch_size']     # N1 full ("everything")

# Fixed, untuned settings. GBR = the exact N1 settings.
MODELS = {
    'GBR (N1)': lambda s: GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                                    subsample=0.8, min_samples_leaf=5, random_state=s),
    'Ridge': lambda s: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'RandomForest': lambda s: RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                    n_jobs=-1, random_state=s),
    'LightGBM': lambda s: LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=15,
                                        min_child_samples=20, subsample=0.8, subsample_freq=1,
                                        random_state=s, verbose=-1),
}

def load(csv, targets):
    df = pd.read_csv(csv)
    assert len(df) == 1600, f'expected 1600 rows, got {len(df)}'
    assert (df['benchmark_version'].astype(str) == '1.2').all(), 'not a v1.2 benchmark'
    need = INPUTS + list(targets.values())
    assert df[need].isna().sum().sum() == 0, 'NaN in inputs or targets'
    df['group'] = np.where(df['drift_type'] == 'none', df['condition_id'],
                           df['condition_id'].str.rsplit('_', n=1).str[0])
    print(f"  rows {len(df)} | groups {df['group'].nunique()} (expected 480)")
    return df

def run(name, cfg):
    print(f'\n=== {name.upper()} ===')
    df = load(cfg['csv'], cfg['targets'])
    X = df[INPUTS].values.astype(float)
    n = len(df)
    oof = {t: {m: np.zeros((REPEATS, n)) for m in ['constant'] + list(MODELS)} for t in cfg['targets']}
    secs = {m: 0.0 for m in MODELS}
    t0 = time.time()
    for r in range(REPEATS):
        folds = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=r)
                     .split(df, df['drift_type'], df['group']))      # same folds for every model
        for t, col in cfg['targets'].items():
            y = df[col].values
            for k, (tr, te) in enumerate(folds):
                oof[t]['constant'][r, te] = y[tr].mean()
                for m, make in MODELS.items():
                    s = time.time()
                    oof[t][m][r, te] = make(1000 * r + k).fit(X[tr], y[tr]).predict(X[te])
                    secs[m] += time.time() - s
        print(f'  repeat {r + 1}/{REPEATS} ({(time.time() - t0) / 60:.1f} min)', flush=True)
    return df, oof, secs

## 3. Run both datasets

In [ ]:
RAW = {}
for name, cfg in DATASETS.items():
    RAW[name] = run(name, cfg)


=== UCI ===
  rows 1600 | groups 480 (expected 480)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

  repeat 1/10 (1.7 min)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

  repeat 2/10 (3.3 min)


...

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

  repeat 9/10 (14.3 min)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

  repeat 10/10 (15.9 min)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 4. Results
- **MAE** (pp, lower is better), mean +/- sd over the 10 repeats; R2.
- **vs GBR**: the MAE difference (model minus GBR; positive = GBR better) and in how many of the 10 repeats GBR had the lower MAE.

In [ ]:
RESULTS = {}
for name, cfg in DATASETS.items():
    df, oof, secs = RAW[name]
    RESULTS[name] = {}
    for t, col in cfg['targets'].items():
        y = df[col].values
        mae = {m: np.array([mean_absolute_error(y, oof[t][m][r]) for r in range(REPEATS)]) for m in oof[t]}
        rows = []
        for m in oof[t]:
            r2 = np.mean([r2_score(y, oof[t][m][r]) for r in range(REPEATS)])
            rmse = np.mean([np.sqrt(mean_squared_error(y, oof[t][m][r])) for r in range(REPEATS)])
            diff = mae[m] - mae['GBR (N1)']
            rows.append({'model': m, 'MAE': round(mae[m].mean(), 3), 'sd': round(mae[m].std(), 3),
                         'RMSE': round(rmse, 3), 'R2': round(r2, 3),
                         'minus GBR': round(diff.mean(), 3),
                         'GBR better (of 10)': int((diff > 0).sum()) if m != 'GBR (N1)' else None,
                         'fit time (s, all folds)': round(secs.get(m, 0.0), 1) if m in secs else None})
        tab = pd.DataFrame(rows).sort_values('MAE')
        RESULTS[name][t] = tab.to_dict(orient='records')
        print(f"\n{name.upper()} | target {t} ({col})")
        print(tab.to_string(index=False))
        rec = cfg['recorded_gbr'][t]
        got = mae['GBR (N1)'].mean()
        print(f"  sanity: GBR {got:.3f} vs recorded {rec:.3f} -> {'OK' if abs(got - rec) < 0.01 else 'CHECK: differs from the recorded run'}")


UCI | target fixed (actual_accuracy_drop)
       model   MAE    sd  RMSE     R2  minus GBR  GBR better (of 10)  fit time (s, all folds)
    GBR (N1) 0.903 0.010 1.347  0.927      0.000                 NaN                    194.5
RandomForest 0.951 0.011 1.709  0.882      0.048                10.0                     86.4
    LightGBM 1.096 0.019 2.142  0.814      0.193                10.0                      8.0
       Ridge 1.173 0.006 1.695  0.884      0.269                10.0                      0.3
    constant 2.138 0.004 4.988 -0.003      1.235                10.0                      NaN
  sanity: GBR 0.903 vs recorded 0.903 -> OK

UCI | target paired (harm_paired)
       model   MAE    sd  RMSE     R2  minus GBR  GBR better (of 10)  fit time (s, all folds)
    GBR (N1) 0.657 0.010 1.115  0.948      0.000                 NaN                    194.5
RandomForest 0.691 0.015 1.543  0.901      0.034                10.0                     86.4
    LightGBM 0.847 0.022 2.002  

## 5. MAE by drift type (fixed target)

In [ ]:
for name, cfg in DATASETS.items():
    df, oof, _ = RAW[name]
    col = cfg['targets']['fixed']; y = df[col].values
    out = {}
    for m in oof['fixed']:
        out[m] = {dt: np.mean([mean_absolute_error(y[mask], oof['fixed'][m][r][mask]) for r in range(REPEATS)])
                  for dt in sorted(df['drift_type'].unique()) for mask in [(df['drift_type'] == dt).values]}
    print(f'\n{name.upper()} (fixed target, MAE pp by drift type)')
    print(pd.DataFrame(out).round(3).to_string())


UCI (fixed target, MAE pp by drift type)
                 constant  GBR (N1)  Ridge  RandomForest  LightGBM
combined            1.304     0.862  1.101         0.852     0.917
interaction         1.411     0.770  0.925         0.762     0.789
marginal_multi      3.030     1.201  1.585         1.286     1.639
marginal_single     2.658     0.918  1.258         1.030     1.225
none                1.842     0.680  0.818         0.682     0.689

IEEE (fixed target, MAE pp by drift type)
                 constant  GBR (N1)  Ridge  RandomForest  LightGBM
combined            1.226     1.001  1.000         0.997     1.037
interaction         1.101     0.883  0.889         0.886     0.888
marginal_multi      2.508     1.205  1.123         1.179     1.205
marginal_single     1.049     1.035  1.001         1.027     1.034
none                1.080     0.862  0.839         0.836     0.871


## 6. Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
path = f'{OUT_DIR}/novelty1_model_comparison_v12.json'
json.dump({'protocol': {'repeats': REPEATS, 'folds': FOLDS, 'cv': 'StratifiedGroupKFold by drift_type, one group per condition',
                        'inputs': INPUTS, 'tuned': False,
                        'models': {'GBR (N1)': 'n_estimators 200, max_depth 4, lr 0.05, subsample 0.8, min_samples_leaf 5',
                                   'Ridge': 'StandardScaler + Ridge(alpha 1.0)',
                                   'RandomForest': 'n_estimators 300, min_samples_leaf 5',
                                   'LightGBM': 'n_estimators 200, lr 0.05, num_leaves 15, min_child_samples 20, subsample 0.8'}},
           'benchmarks': {k: os.path.basename(v['csv']) for k, v in DATASETS.items()},
           'results': RESULTS}, open(path, 'w'), indent=2)
print('saved', path)

saved /content/drive/MyDrive/J26-DS-361-Datasets/c1_models_v12/novelty1_model_comparison_v12.json


## How to read the outcome
- **GBR lowest or within about 0.02 pp of the best, on both datasets:** keep GBR. Thesis line: *"GBR was compared with Ridge, Random Forest and LightGBM under the same group-aware CV and was best (or tied) on both datasets."*
- **Ridge clearly worse:** the harm relationship is non-linear, which justifies a tree ensemble.
- **Another model clearly better on both datasets (lower MAE in 9 or 10 of 10 repeats):** report it honestly. Do not switch the frozen PP1 engine; note it as a PP2 option.